In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
import pandas as pd
import numpy as np
import glob
import os
import json
from tqdm.auto import tqdm

# global CTR
data_dir = "/kaggle/input/datasets/huy291/criteo-cleaned-data/data"
parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))

total_clicks = 0
total_impressions = 0

print("Bước 1: Quét Global CTR...")
for f in tqdm(parquet_files, desc="Calc Global CTR"):
    df = pd.read_parquet(f, columns=["label"])
    total_clicks += df["label"].sum()
    total_impressions += len(df)

ctr_global = total_clicks / total_impressions
print(f"Global CTR: {ctr_global:.4f}")

# ctr per id
stats = {}
cat_cols = [f"C{i}" for i in range(1, 27)]

print("Bước 2: Gom nhóm và đếm tần suất ID...")
for f in tqdm(parquet_files, desc="Aggregate IDs"):
    df = pd.read_parquet(f, columns=["label"] + cat_cols)
    
    # melt 26 cột thành 1 cột
    df_melted = df.melt(id_vars=["label"], value_vars=cat_cols, var_name="Feature_Name", value_name="Raw_Value")
    df_melted = df_melted.dropna(subset=["Raw_Value"])
    df_melted["Global_ID"] = df_melted["Feature_Name"] + "_" + df_melted["Raw_Value"].astype(str)
    
    # aggregate
    agg = df_melted.groupby("Global_ID").agg(N=("label", "count"), Clicks=("label", "sum"))
    
    # dictionary
    for gid, row in agg.iterrows():
        if gid not in stats:
            stats[gid] = [0, 0]
        stats[gid][0] += row["N"]
        stats[gid][1] += row["Clicks"]

# ivs and tame
print("Bước 3: Tính IVS và phân bổ không gian (Pareto 80/20)...")
ivs_records = []
for gid, (n, clicks) in stats.items():
    ctr_id = clicks / n
    ivs = np.log10(n + 1) * abs(ctr_id - ctr_global)
    ivs_records.append({"Global_ID": gid, "N": n, "IVS": ivs})

df_ivs = pd.DataFrame(ivs_records).sort_values("IVS", ascending=False)
total_ivs = df_ivs["IVS"].sum()
df_ivs["Cum_Percent"] = df_ivs["IVS"].cumsum() / total_ivs

def assign_group(p):
    if p <= 0.50: return 64
    elif p <= 0.70: return 32
    elif p <= 0.85: return 16
    elif p <= 0.95: return 8
    elif p <= 0.99: return 4
    else: return 2

df_ivs["Dim_Group"] = df_ivs["Cum_Percent"].apply(assign_group)

# Local Index (bắt đầu từ 1, 0 cho Padding)
df_ivs["Local_Index"] = df_ivs.groupby("Dim_Group").cumcount() + 1

mapping_dict = df_ivs.set_index("Global_ID")[["Dim_Group", "Local_Index"]].to_dict('index')
vocab_sizes = df_ivs.groupby("Dim_Group").size().to_dict()
vocab_sizes = {str(k): int(v) for k, v in vocab_sizes.items()}

with open("tame_ivs_dictionary.json", "w") as f:
    json.dump(mapping_dict, f)

with open("tame_vocab_sizes.json", "w") as f:
    json.dump(vocab_sizes, f)

print(f"Hoàn tất! Tổng số ID: {len(mapping_dict)}")
print("Kích thước từ vựng các nhóm:", vocab_sizes)

Bước 1: Quét Global CTR...


Calc Global CTR:   0%|          | 0/50 [00:00<?, ?it/s]

Global CTR: 0.2562
Bước 2: Gom nhóm và đếm tần suất ID...


Aggregate IDs:   0%|          | 0/50 [00:00<?, ?it/s]

Bước 3: Tính IVS và phân bổ không gian (Pareto 80/20)...
Hoàn tất! Tổng số ID: 33762577
Kích thước từ vựng các nhóm: {'2': 1078813, '4': 2194850, '8': 5487124, '16': 8230687, '32': 8112045, '64': 8659058}


In [ ]:
import math
import json
import torch
from torch.utils.data import Dataset

class TAMECriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols, dict_path):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        self.supported_dims = ["64", "32", "16", "8", "4", "2"]
        
        print("Đang tải TAME Dictionary vào bộ nhớ...")
        with open(dict_path, "r") as f:
            self.tame_dict = json.load(f)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        # dense
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                dense_x.append(math.log1p(float(val))) 
        
        # categorical (Positional Routing)
        num_cats = len(self.cat_cols)
        # Tạo sẵn 6 vector, mỗi vector có đúng 26 slot số 0 (Vector rỗng)
        grouped_cat = {dim: [0] * num_cats for dim in self.supported_dims}
        
        for col_idx, col in enumerate(self.cat_cols):
            val = row[col]
            if val is not None:
                gid = f"{col}_{val}"
                if gid in self.tame_dict:
                    info = self.tame_dict[gid]
                    dim_group = str(info["Dim_Group"])
                    local_idx = info["Local_Index"]
                    
                    # Cắm ID vào đúng vị trí cột gốc của nó
                    grouped_cat[dim_group][col_idx] = local_idx
        
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            {dim: torch.tensor(grouped_cat[dim], dtype=torch.long) for dim in self.supported_dims},
            torch.tensor(label, dtype=torch.float32)
        )

In [ ]:
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, random_split
import glob
import os

dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

print("Đang tải dữ liệu parquet local...")
data_dir = "/kaggle/input/datasets/huy291/criteo-cleaned-data/data"

parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))

print(f"Số file parquet: {len(parquet_files)}")

# load bằng HuggingFace datasets
raw_dataset = load_dataset(
    "parquet",
    data_files=parquet_files,
    split="train"
)

print(raw_dataset)

print("Đang khởi tạo Dataset và DataLoader...")
tame_dataset = TAMECriteoDataset(raw_dataset, dense_cols, cat_cols, "tame_ivs_dictionary.json")

train_size = int(0.9 * len(tame_dataset))
val_size = len(tame_dataset) - train_size
train_dataset, val_dataset = random_split(tame_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset, 
    batch_size=4096, 
    shuffle=True, 
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=4096, 
    shuffle=False, 
    num_workers=4, 
    persistent_workers=True, 
    pin_memory=True
)
print("Hoàn tất DataLoader!")

Đang tải dữ liệu parquet local...
Số file parquet: 50


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/34 [00:00<?, ?it/s]

Dataset({
    features: ['label', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'I10', 'I11', 'I12', 'I13', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26'],
    num_rows: 45840617
})
Đang khởi tạo Dataset và DataLoader...
Đang tải TAME Dictionary vào bộ nhớ...
Hoàn tất DataLoader!


In [5]:
import torch
import torch.nn as nn

class TAME_Embedding(nn.Module):
    """
    TAME: Target-Aware Memory-efficient Embedding Layer.
    Sử dụng Positional Sum-Routing để chập 6 không gian chiều lại làm 1,
    loại bỏ hoàn toàn sự lạm phát tham số.
    """
    def __init__(self, vocab_sizes, target_dim=64):
        super(TAME_Embedding, self).__init__()
        self.target_dim = target_dim
        self.supported_dims = [64, 32, 16, 8, 4, 2]
        self.embedding_blocks = nn.ModuleDict()
        
        for dim in self.supported_dims:
            dim_str = str(dim)
            if dim_str in vocab_sizes and vocab_sizes[dim_str] > 0:
                self.embedding_blocks[dim_str] = nn.Embedding(
                    num_embeddings=vocab_sizes[dim_str] + 1, 
                    embedding_dim=dim,
                    padding_idx=0
                )

    def forward(self, grouped_inputs):
        final_output = 0
        
        for dim in self.supported_dims:
            dim_str = str(dim)
            if dim_str in grouped_inputs and dim_str in self.embedding_blocks:
                x = grouped_inputs[dim_str]
                emb = self.embedding_blocks[dim_str](x)
                
                # Zero-Flop Tiling
                if dim == self.target_dim:
                    out = emb
                else:
                    n_repeats = self.target_dim // dim
                    out = emb.repeat(1, 1, n_repeats) / n_repeats
                    
                # cộng dồn
                final_output = final_output + out
        
        # output [Batch, 26, 64]
        return final_output

In [6]:
import torch
import torch.nn as nn

class TAME_DeepFM(nn.Module):
    def __init__(self, vocab_sizes, num_dense_features, target_dim=64, hidden_dims=[256, 128, 64]):
        super(TAME_DeepFM, self).__init__()
        
        self.num_dense_features = num_dense_features
        self.num_cat_features = 26 
        
        # tame
        self.tame_emb = TAME_Embedding(vocab_sizes, target_dim=target_dim)
        
        #  FM bậc 1 (linear component)
        # linear(64, 1) để nén vector emb thành trọng số bậc 1
        self.fm_1st_sparse = nn.Linear(target_dim, 1)
        
        if num_dense_features > 0:
            self.fm_1st_dense = nn.Linear(num_dense_features, 1)
            
        # deep (MLP)
        deep_in_dim = (self.num_cat_features * target_dim) + num_dense_features
        deep_layers = []
        in_dim = deep_in_dim
        for dim in hidden_dims:
            deep_layers.append(nn.Linear(in_dim, dim))
            deep_layers.append(nn.BatchNorm1d(dim)) # batchnorm
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.Dropout(0.2))
            in_dim = dim
            
        deep_layers.append(nn.Linear(in_dim, 1))
        self.deep_net = nn.Sequential(*deep_layers)

    def forward(self, dense_x, grouped_cat_x):
        # [Batch, 26, 64]
        emb_tensor = self.tame_emb(grouped_cat_x)
        
        # [Batch, 26, 64] -> [Batch, 26, 1], sau đó cộng tổng 26 cột lại -> [Batch, 1]
        fm_1st_cat = torch.sum(self.fm_1st_sparse(emb_tensor), dim=1)
        
        if self.num_dense_features > 0:
            fm_1st = fm_1st_cat + self.fm_1st_dense(dense_x)
        else:
            fm_1st = fm_1st_cat
            
        #  bậc 2
        sum_of_emb = torch.sum(emb_tensor, dim=1)         # [Batch, 64]
        sum_of_emb_sq = sum_of_emb ** 2                   # [Batch, 64]
        sq_of_emb = emb_tensor ** 2                       # [Batch, 26, 64]
        sum_of_sq_emb = torch.sum(sq_of_emb, dim=1)       # [Batch, 64]
        
        fm_2nd = 0.5 * torch.sum(sum_of_emb_sq - sum_of_sq_emb, dim=1, keepdim=True) # [Batch, 1]
        
        #  Deep Component
        # [Batch, 26, 64] to [Batch, 1664]
        deep_in_cat = emb_tensor.view(emb_tensor.size(0), -1) 
        
        if self.num_dense_features > 0:
            deep_in = torch.cat([dense_x, deep_in_cat], dim=1) # [Batch, 1677]
            
        deep_out = self.deep_net(deep_in) # [Batch, 1]
        
        # Combine
        out = fm_1st + fm_2nd + deep_out
        
        return out.squeeze(1)

In [7]:
import json

with open("tame_vocab_sizes.json", "r") as f:
    vocab_sizes = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

model = TAME_DeepFM(vocab_sizes=vocab_sizes, num_dense_features=len(dense_cols))

if num_gpus > 1:
    model = nn.DataParallel(model)

model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

print("Bắt đầu huấn luyện TAME_DeepFM...")
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_targets, train_preds = [], []

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")
    for dense_x, grouped_cat_x, labels in train_bar:
        dense_x = dense_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)
        
        # Push dict of tensors to GPU
        grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

        optimizer.zero_grad()
        logits = model(dense_x, grouped_cat_x)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        probs = torch.sigmoid(logits)
        train_targets.extend(labels.detach().cpu().numpy())
        train_preds.extend(probs.detach().cpu().numpy())
        
        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)

    # val
    model.eval()
    val_loss = 0.0
    val_targets, val_preds = [], []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, grouped_cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)
            grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

            logits = model(dense_x, grouped_cat_x)
            loss = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)

    print(
        f"Epoch {epoch+1}: \n"
        f"Train Loss: {avg_train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition
Bắt đầu huấn luyện TAME_DCN...


[TRAIN] Epoch 1/3:   0%|          | 0/10073 [00:02<?, ?it/s]

[VAL] Epoch 1/3:   0%|          | 0/1120 [00:00<?, ?it/s]

Epoch 1: 
Train Loss: 1.8219 | Train AUC: 0.7456 || Val Loss: 0.7333 | Val AUC: 0.7925



[TRAIN] Epoch 2/3:   0%|          | 0/10073 [00:01<?, ?it/s]

[VAL] Epoch 2/3:   0%|          | 0/1120 [00:00<?, ?it/s]

Epoch 2: 
Train Loss: 0.7769 | Train AUC: 0.7870 || Val Loss: 0.6623 | Val AUC: 0.8062



[TRAIN] Epoch 3/3:   0%|          | 0/10073 [00:01<?, ?it/s]

[VAL] Epoch 3/3:   0%|          | 0/1120 [00:00<?, ?it/s]

Epoch 3: 
Train Loss: 0.7660 | Train AUC: 0.7884 || Val Loss: 0.6930 | Val AUC: 0.7842

